# Data Preparation

Cleaning steps decided in the initial analysis. The cleaned data is saved to `data/processed/`.

In [1]:
import pandas as pd

df_nightly = pd.read_csv('../data/raw/sleep_nightly.csv')
df_scores = pd.read_csv('../data/raw/sleep_scores.csv')

df = pd.merge(df_nightly, df_scores, how='inner', on='night')

## Drop Columns

In [2]:
df = df.drop(columns=['bedtime_y', 'awake_min_y', 'notes'])

## Rename Columns

In [3]:
df = df.rename(columns={'bedtime_x': 'bedtime', 'awake_min_x': 'awake_min'})

## Drop Rows

In [4]:
df = df.dropna(subset=['interruption_pts', 'wake_ups'])

## Transform Columns

In [5]:
df['night'] = pd.to_datetime(df['night'], format='%Y-%m-%d')
df['bedtime'] = pd.to_datetime(df['bedtime'], format='%Y-%m-%d %H:%M')
df['wake_time'] = pd.to_datetime(df['wake_time'], format='%Y-%m-%d %H:%M')

In [6]:
df['wake_ups'] = df['wake_ups'].astype('int')

## What Was Done

- Dropped columns: *bedtime_y*, *awake_min_y* and *notes*, as they were either duplicate columns or too many missing values.
- Renamed columns: *bedtime_x* to *bedtime*, *awake_min_x* to *awake_min*.
- Dropped rows where *interruption_pts* and *wake_ups* are missing (2 rows).
- Converted *night*, *bedtime* (former *bedtime_x*) and *wake_time* to datetime formats.
- Converted *wake_ups* to integer (it was stored as float because of missing values).

In [7]:
df.to_csv('../data/processed/sleep_data.csv', index=False)

# Sleep Records

To see *when* during the night I wake up, I need the individual records from `sleep_records.csv`. Only the `SleepAnalysis` rows describe sleep stages (Core, Deep, REM, Awake, Unspecified); the other two types are wrist temperature and breathing disturbances, which I don't use.

In [8]:
df_records = pd.read_csv('../data/raw/sleep_records.csv')

df_records = df_records[df_records['type'] == 'SleepAnalysis']
df_records = df_records.drop(columns=['type', 'unit', 'source'])
df_records = df_records.rename(columns={'value': 'stage'})

df_records['night'] = pd.to_datetime(df_records['night'], format='%Y-%m-%d')
df_records['start'] = pd.to_datetime(df_records['start'], format='%Y-%m-%d %H:%M:%S')
df_records['end'] = pd.to_datetime(df_records['end'], format='%Y-%m-%d %H:%M:%S')

df_records['stage'].value_counts()

stage
Core           7008
Awake          4277
REM            2707
Deep           2177
Unspecified      80
Name: count, dtype: int64

## What Was Done

- Kept only the `SleepAnalysis` rows (16,249 of 17,075).
- Dropped columns: *type* (now always SleepAnalysis), *unit* (empty for these rows) and *source* (the same on every row).
- Renamed *value* to *stage*, since for these rows it holds the sleep stage name.
- Converted *night*, *start* and *end* to datetime formats.

In [9]:
df_records.to_csv('../data/processed/sleep_records.csv', index=False)